In [1]:
from mace.calculators import MACECalculator

from ase.io import read, write
from ase.build import graphene
from ase import Atom, Atoms

from ase.phonons import Phonons

from ase.optimize import BFGS
from ase.filters import ExpCellFilter

from ase.visualize import view

import numpy as np
import matplotlib.pyplot as plt

import os

from tqdm import tqdm

from os import walk

from multiprocessing import Pool

/home/gabrielecolombo/Thesis/PROJECT/.venv/lib/python3.12/site-packages/e3nn/o3/_wigner.py:10: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  _Jd, _W3j_flat, _W3j_indices = torch.load(os.path.join(os.path.dirname(__file__), 'constants.pt'))
/home/gabrielecolombo/Thesis/PROJECT/.venv/lib/python3.12/site-packages/torch/jit/_script.py:1488: DeprecationWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


In [2]:
model = MACECalculator("../../MACE.model", device="cuda")

/home/gabrielecolombo/Thesis/PROJECT/.venv/lib/python3.12/site-packages/mace/calculators/mace.py:197: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  torch.load(f=model_path, map_location=device)


Using head Default out of ['Default']
No dtype selected, switching to float32 to match model dtype.


/home/gabrielecolombo/Thesis/PROJECT/.venv/lib/python3.12/site-packages/torch/jit/_serialization.py:176: DeprecationWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(
/home/gabrielecolombo/Thesis/PROJECT/.venv/lib/python3.12/site-packages/torch/jit/_serialization.py:176: DeprecationWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(
/home/gabrielecolombo/Thesis/PROJECT/.venv/lib/python3.12/site-packages/torch/jit/_serialization.py:176: DeprecationWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(


In [3]:
atoms = graphene()
atoms.cell[2] = [0, 0, 100]
atoms.calc = model

filter = ExpCellFilter(atoms, mask=[1,1,0,0,0,0])
opt = BFGS(filter)
opt.run(fmax = 1e-5)

/tmp/ipykernel_9673/1559521758.py:5: DeprecationWarning: Use FrechetCellFilter for better convergence w.r.t. cell variables.
  filter = ExpCellFilter(atoms, mask=[1,1,0,0,0,0])


      Step     Time          Energy          fmax
BFGS:    0 11:42:38      -15.917213        0.014590
BFGS:    1 11:42:38      -15.917210        0.029133
BFGS:    2 11:42:38      -15.917213        0.000013
BFGS:    3 11:42:38      -15.917213        0.000023
BFGS:    4 11:42:38      -15.917217        0.000034
BFGS:    5 11:42:38      -15.917215        0.000026
BFGS:    6 11:42:38      -15.917213        0.000007


np.True_

In [4]:
DEG2RAD = np.pi / 180
def rotate_atoms(atoms, rot):
    """Return a new Atoms object with positions and cell rotated by rot (radians) around Z."""
    atoms = atoms.copy()
    rot = rot * DEG2RAD
    R = np.array([
        [np.cos(rot), -np.sin(rot), 0],
        [np.sin(rot),  np.cos(rot), 0],
        [0,            0,           1],
    ])
    new_cell = atoms.cell @ R.T
    atoms.set_cell(new_cell, scale_atoms=False)
    new_positions = atoms.positions @ R.T
    atoms.set_positions(new_positions)
    return atoms

In [5]:
atoms.cell[2,2] = 100

In [6]:
strained_atoms = atoms.copy()
cell0 = strained_atoms.cell.copy()

In [7]:
strain = np.array([
    [0.2, 0, 0],
    [0,0.2,0],
    [0,0,0]
    ])
deformation = np.eye(3) + strain
cell = cell0 @ deformation
strained_atoms.set_cell(cell, scale_atoms=True)
view(rotate_atoms(strained_atoms.repeat((5,5,1)), 90), viewer="x3d")

In [8]:
strained_atoms.calc = model
strained_atoms.get_stress()

array([ 2.9289117e-02,  2.9289119e-02,  4.9045446e-44,  1.1275466e-30,
       -2.0460737e-30,  5.5369875e-10], dtype=float32)

In [9]:
OUTPHONOS="PHONONS_ROTATED"
os.makedirs(OUTPHONOS, exist_ok=True)

def plot_phonons(bs, dos, outdir):
    # print("[INFO] Plotting")
    # print("[INFO] Results will be saved to", outdir)
    fig = plt.figure(figsize=(7, 4))
    ax = fig.add_axes([0.12, 0.07, 0.67, 0.85])

    emax = bs.energies.max()
    bs.plot(ax=ax, emin=0.0, emax=emax)

    dosax = fig.add_axes([0.8, 0.07, 0.17, 0.85])
    dosax.fill_between(
        dos.get_weights(),
        dos.get_energies(),
        y2=0,
        color="grey",
        edgecolor="k",
        lw=1,
    )
    dosax.set_ylim(0, emax)
    dosax.set_yticks([])
    dosax.set_xticks([])
    dosax.set_xlabel("DOS")

    bs.write(os.path.join(outdir, "bs.json"))

    energies = dos.get_energies()
    weights = dos.get_weights()

    # Save to a simple compressed or text file
    np.savez(os.path.join(outdir, "dos.npz"), energy=energies, weights=weights)

    outfig = os.path.join(outdir, "phonons.svg")
    # print("saving fig to", outfig)
    fig.savefig(outfig)
    plt.close(fig)

In [10]:
def compute_phonons(strain_tensor, OUTDIR, name):

    strained_atoms = atoms.copy()
    strained_atoms.calc = model

    outdir = os.path.join(OUTPHONOS, OUTDIR, name)
    os.makedirs(outdir, exist_ok=True)
    deformation = np.eye(3) + strain_tensor
    cell = cell0 @ deformation
    strained_atoms.set_cell(cell, scale_atoms=True)

    strained_atoms.calc = model

    opt = BFGS(strained_atoms, logfile=None)
    opt.run(fmax=1e-4)
    outconfig = os.path.join(outdir, "relaxed.extxyz")
    stress = strained_atoms.get_stress()
    e = strained_atoms.get_potential_energy()
    fs = strained_atoms.get_forces()
    write(outconfig, strained_atoms)

    ph = Phonons(strained_atoms, model, supercell=(10, 10, 1), delta=0.01)

    # print("Running...")
    ph.run()
    # print("Reading...")
    ph.read(acoustic=True)
    ph.clean()
    path = atoms.cell.bandpath("GKMG", npoints=200)
    # print("BS and DOS")
    bs = ph.get_band_structure(path)
    dos = ph.get_dos(kpts=(30, 30, 1)).sample_grid(npts=200, width=1e-3)

    # print("PLOTTING")
    plot_phonons(bs, dos, outdir)

In [16]:
strainversos = np.array([[np.sqrt(2)/2, 0, 0], [0, np.sqrt(2)/2, 0], [0, 0, 0]])

In [19]:
compute_phonons(0.227 * strainversos, "PLOTFORPRESENTATION", "0.2")

WARNING, 2 imaginary frequencies at q = ( 0.00,  0.00,  0.00) ; (omega_q = 3.987e-03*i)
WARNING, 2 imaginary frequencies at q = ( 0.00,  0.00,  0.00) ; (omega_q = 3.987e-03*i)


/tmp/ipykernel_9673/643072565.py:37: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.savefig(outfig)


In [47]:
uniaxial_strains = np.linspace(0, 0.40, 100)
uniaxial_strains_tensors = [
    np.array([[us, 0, 0], 
              [0, 0, 0], 
              [0, 0, 0]]) for us in uniaxial_strains
]

names = ["us_" + str(us) for us in uniaxial_strains]

for us, name in tqdm(zip(uniaxial_strains_tensors[:], names)):
    compute_phonons(us, "US_ZIGZAG", name)

0it [00:00, ?it/s]/tmp/ipykernel_2142129/643072565.py:37: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.savefig(outfig)
1it [00:00,  1.43it/s]

WARNING, 1 imaginary frequencies at q = ( 0.00,  0.00,  0.00) ; (omega_q = 1.746e-03*i)
WARNING, 1 imaginary frequencies at q = ( 0.00,  0.00,  0.00) ; (omega_q = 1.746e-03*i)
WARNING, 1 imaginary frequencies at q = (-0.02,  0.02,  0.00) ; (omega_q = 2.336e-03*i)
WARNING, 1 imaginary frequencies at q = ( 0.02, -0.02,  0.00) ; (omega_q = 2.336e-03*i)


/tmp/ipykernel_2142129/643072565.py:37: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.savefig(outfig)
2it [00:01,  1.39it/s]

WARNING, 1 imaginary frequencies at q = ( 0.00,  0.00,  0.00) ; (omega_q = 4.264e-03*i)
WARNING, 1 imaginary frequencies at q = ( 0.00,  0.00,  0.00) ; (omega_q = 4.264e-03*i)


/tmp/ipykernel_2142129/643072565.py:37: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.savefig(outfig)
3it [00:02,  1.32it/s]

...

WARNING, 2 imaginary frequencies at q = ( 0.00,  0.00,  0.00) ; (omega_q = 5.097e-03*i)
WARNING, 2 imaginary frequencies at q = ( 0.00,  0.00,  0.00) ; (omega_q = 5.097e-03*i)
WARNING, 1 imaginary frequencies at q = (-0.48, -0.48,  0.00) ; (omega_q = 2.354e-01*i)
WARNING, 1 imaginary frequencies at q = (-0.48, -0.45,  0.00) ; (omega_q = 2.306e-01*i)
WARNING, 1 imaginary frequencies at q = (-0.48, -0.42,  0.00) ; (omega_q = 2.206e-01*i)
WARNING, 1 imaginary frequencies at q = (-0.48, -0.38,  0.00) ; (omega_q = 2.027e-01*i)
WARNING, 1 imaginary frequencies at q = (-0.48, -0.35,  0.00) ; (omega_q = 1.720e-01*i)
WARNING, 1 imaginary frequencies at q = (-0.48, -0.32,  0.00) ; (omega_q = 1.144e-01*i)
WARNING, 1 imaginary frequencies at q = (-0.48,  0.28,  0.00) ; (omega_q = 9.084e-02*i)
WARNING, 1 imaginary frequencies at q = (-0.48,  0.32,  0.00) ; (omega_q = 1.586e-01*i)
WARNING, 1 imaginary frequencies at q = (-0.48,  0.35,  0.00) ; (omega_q = 1.930e-01*i)
WARNING, 1 imaginary frequencies

/tmp/ipykernel_2142129/643072565.py:37: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.savefig(outfig)
99it [01:22,  1.23it/s]

WARNING, 1 imaginary frequencies at q = ( 0.00,  0.00,  0.00) ; (omega_q = 1.513e-03*i)
WARNING, 1 imaginary frequencies at q = ( 0.00,  0.00,  0.00) ; (omega_q = 1.513e-03*i)
WARNING, 1 imaginary frequencies at q = (-0.48, -0.48,  0.00) ; (omega_q = 2.317e-01*i)
WARNING, 1 imaginary frequencies at q = (-0.48, -0.45,  0.00) ; (omega_q = 2.267e-01*i)
WARNING, 1 imaginary frequencies at q = (-0.48, -0.42,  0.00) ; (omega_q = 2.163e-01*i)
WARNING, 1 imaginary frequencies at q = (-0.48, -0.38,  0.00) ; (omega_q = 1.977e-01*i)
WARNING, 1 imaginary frequencies at q = (-0.48, -0.35,  0.00) ; (omega_q = 1.656e-01*i)
WARNING, 1 imaginary frequencies at q = (-0.48, -0.32,  0.00) ; (omega_q = 1.037e-01*i)
WARNING, 1 imaginary frequencies at q = (-0.48,  0.28,  0.00) ; (omega_q = 7.778e-02*i)
WARNING, 1 imaginary frequencies at q = (-0.48,  0.32,  0.00) ; (omega_q = 1.520e-01*i)
WARNING, 1 imaginary frequencies at q = (-0.48,  0.35,  0.00) ; (omega_q = 1.880e-01*i)
WARNING, 1 imaginary frequencies

/tmp/ipykernel_2142129/643072565.py:37: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.savefig(outfig)
100it [01:23,  1.20it/s]

WARNING, 1 imaginary frequencies at q = ( 0.00,  0.00,  0.00) ; (omega_q = 2.000e-03*i)
WARNING, 1 imaginary frequencies at q = ( 0.00,  0.00,  0.00) ; (omega_q = 2.000e-03*i)
WARNING, 1 imaginary frequencies at q = (-0.48, -0.48,  0.00) ; (omega_q = 2.281e-01*i)
WARNING, 1 imaginary frequencies at q = (-0.48, -0.45,  0.00) ; (omega_q = 2.229e-01*i)
WARNING, 1 imaginary frequencies at q = (-0.48, -0.42,  0.00) ; (omega_q = 2.120e-01*i)
WARNING, 1 imaginary frequencies at q = (-0.48, -0.38,  0.00) ; (omega_q = 1.927e-01*i)
WARNING, 1 imaginary frequencies at q = (-0.48, -0.35,  0.00) ; (omega_q = 1.591e-01*i)
WARNING, 1 imaginary frequencies at q = (-0.48, -0.32,  0.00) ; (omega_q = 9.195e-02*i)
WARNING, 1 imaginary frequencies at q = (-0.48,  0.28,  0.00) ; (omega_q = 6.237e-02*i)
WARNING, 1 imaginary frequencies at q = (-0.48,  0.32,  0.00) ; (omega_q = 1.453e-01*i)
WARNING, 1 imaginary frequencies at q = (-0.48,  0.35,  0.00) ; (omega_q = 1.830e-01*i)
WARNING, 1 imaginary frequencies

In [49]:
from ase.units import GPa

def process_file(filepath):
    """Extracts cell lengths and stress from an extxyz file."""
    try:
        atoms = read(filepath)
        # get_cell().lengths() returns [a, b, c]
        # get_stress() returns [sxx, syy, szz, syz, sxz, sxy] in eV/A^3
        return atoms.get_cell().lengths(), atoms.get_stress()
    except Exception as e:
        print(f"Error processing {filepath}: {e}")
        return None

def plot_stress_strain(directory, direction=0):
    """
    direction: 0 for XX, 1 for YY, 2 for ZZ
    """
    file_paths = []
    for root, _, filenames in os.walk(directory):
        for f in filenames:
            if f.endswith(".extxyz"):
                file_paths.append(os.path.join(root, f))
    
    # Sort files to ensure the sequence is chronological/ordered
    file_paths.sort()

    # Parallel processing
    with Pool(processes=16) as pool:
        results = pool.map(process_file, file_paths)

    # Filter out failed reads and unpack
    results = [r for r in results if r is not None]
    
    # Convert to numpy arrays for easier manipulation
    # results is a list of (lengths, stress_vector)
    lengths = np.array([r[0] for r in results])
    stresses = np.array([r[1] for r in results])

    # 1. Calculate Engineering Strain: epsilon = (L - L0) / L0
    # Assuming the first file in the sorted list is the reference (unstrained) state
    L0 = lengths[0, direction]
    strains = (lengths[:, direction] - L0) / L0

    # 2. Extract Stress and convert to GPa
    # ASE stress is usually defined as -1 * Pressure. 
    # For a tensile test in 'x', we take the first component (index 0).
    # We divide by GPa unit to convert eV/A^3 -> GPa
    stress_val = stresses[:, direction] / GPa

    # 3. Sort by strain (important if files weren't named in order)
    sort_idx = np.argsort(strains)
    strains = strains[sort_idx]
    stress_val = stress_val[sort_idx]

    # 4. Plotting
    plt.figure(figsize=(8, 6))
    plt.plot(strains, stress_val, 'o-', linewidth=2, markersize=4)
    
    # Formatting
    axis_name = ['X', 'Y', 'Z'][direction]
    plt.title(f'Stress-Strain Relation ({axis_name}-direction)')
    plt.xlabel('Engineering Strain $\epsilon$')
    plt.ylabel('Stress $\sigma$ (GPa)')
    plt.grid(True, linestyle='--', alpha=0.7)
    
    plt.tight_layout()
    plt.savefig('stress_strain_plot.png')
    print("Plot saved as stress_strain_plot.png")

In [ ]:
plot_stress_strain("PHONONS/US_ZIGZAG", direction=0)

In [ ]:
equibiaxial = np.linspace(0, 0.25, 500)
equibiaxial_tensors = [
    np.array([[es, 0, 0], 
              [0, es, 0], 
              [0, 0, 0]]) for es in equibiaxial
]

names = ["us_" + str(us) for us in equibiaxial]

for us, name in tqdm(zip(equibiaxial_tensors[:], names)):
    compute_phonons(us, "EQUIBIAXIAL", name)